# Dönerläden Berlin — Google Places Text Search API

**Notebook 1 der Datenbeschaffungs-Pipeline**

Dieses Notebook ruft alle Döner-Läden in Berlin über die **Google Places Text Search (New) API** ab und bereitet die Rohdaten für die weitere Analyse vor.

## Strategie

Die Google Places API gibt maximal 60 Ergebnisse pro Suchanfrage zurück (3 Seiten × 20). Um ganz Berlin abzudecken, wird ein **Grid-Ansatz mit adaptiver Verfeinerung** verwendet:

1. Berlin wird in ein **10×10 Grid** (100 Zellen) über die Bounding Box aufgeteilt
2. Jede Zelle wird mit `textQuery: "Döner Kebab"` und einer `rectangle`-Einschränkung abgefragt
3. Bis zu **3 Seiten à 20 Ergebnisse** pro Zelle werden abgerufen
4. Liefert eine Zelle **60 Ergebnisse** (= Maximum), wird sie **rekursiv in 4 Unter-Zellen** aufgeteilt (bis Tiefe 4)
5. Alle Ergebnisse werden über `place_id` **dedupliziert**

## Outputs

| Datei | Beschreibung |
|-------|--------------|
| `dataset_berlin_doener.json` | Rohdaten von der API (alle 79 Felder, 1.346 Einträge) |
| `dataset_berlin_doener_clean.csv` | Bereinigte Tabelle (26 Spalten) für die BI-Pipeline |

## Kosten

Ca. **184 API-Calls × $0.032 = ~$5.90** (Text Search New, Stand März 2026)

## Schritt 1 — Rohdaten abrufen

Der folgende Code-Block implementiert den Grid-Crawler:
- `text_search_request()`: Einzelner API-Call mit optionalem Pagination-Token
- `search_grid_cell()`: Durchsucht eine Zelle rekursiv (verfeinert wenn 60 Ergebnisse)
- `build_grid()`: Erstellt das 10×10 Startraster mit 5% Overlap zwischen Zellen
- `deduplicate_places()`: Entfernt Duplikate via `place_id`
- `main()`: Orchestriert den gesamten Ablauf und speichert `dataset_berlin_doener.json`

**Hinweis:** Der API-Key wird aus der `.env`-Datei geladen (`GOOGLE_CLOUD_API_KEY`).

In [ ]:
"""
Berlin Dönerläden Finder
========================
Findet alle Dönerläden in Berlin über die Google Places API.

Strategie:
1. Berlin wird in ein Grid unterteilt (mind. 10x10)
2. Pro Grid-Zelle: Text Search (New) mit rectangle locationRestriction
3. Bis zu 3 Seiten à 20 Ergebnisse pro Zelle (= max 60)
4. Falls eine Zelle 60 Ergebnisse liefert → rekursiv in 4 Unter-Zellen aufteilen
5. Deduplizierung über place_id
"""

import json
import time
import math
import os
import requests
from datetime import datetime
from dotenv import load_dotenv

load_dotenv()

# ============================================================
# API KEY (aus .env Datei)
# ============================================================
API_KEY = os.getenv("GOOGLE_CLOUD_API_KEY", "")

# ============================================================
# Konfiguration
# ============================================================
INITIAL_GRID_SIZE = 10          # Startraster
MAX_RESULTS_PER_PAGE = 20       # Google-Limit pro Seite
MAX_PAGES = 3                   # Maximal 3 Seiten pro Request-Kette
RESULTS_TRIGGER_REFINE = 60     # Ab 60 Ergebnissen wird verfeinert
SLEEP_BETWEEN_REQUESTS = 0.3    # Sekunden zwischen API-Calls
MAX_RECURSION_DEPTH = 4         # Maximale Verfeinerungstiefe
OUTPUT_FILE = "dataset_berlin_doener.json"

# Berlin Bounding Box (ungefähr)
BERLIN_SOUTH = 52.3382
BERLIN_NORTH = 52.6755
BERLIN_WEST = 13.0884
BERLIN_EAST = 13.7612

# Berlin Place ID für Aggregate API
BERLIN_PLACE_ID = "ChIJAVkDPzdOqEcRcDteW0YgIQQ"

# ============================================================
# Zähler für API-Calls
# ============================================================
api_call_count = 0


def log(msg):
    """Logging mit Timestamp."""
    print(f"[{datetime.now().strftime('%H:%M:%S')}] {msg}")


def text_search_request(rectangle_low, rectangle_high, page_token=None):
    """
    Führt einen Text Search (New) Request durch.

    Args:
        rectangle_low: dict mit latitude/longitude (SW-Ecke)
        rectangle_high: dict mit latitude/longitude (NE-Ecke)
        page_token: Optional, Token für nächste Seite

    Returns:
        tuple: (list of places, next_page_token or None)
    """
    global api_call_count

    url = "https://places.googleapis.com/v1/places:searchText"

    headers = {
        "Content-Type": "application/json",
        "X-Goog-Api-Key": API_KEY,
        "X-Goog-FieldMask": "places.*,nextPageToken"
    }

    body = {
        "textQuery": "Döner Kebab",
        "locationRestriction": {
            "rectangle": {
                "low": {
                    "latitude": rectangle_low["latitude"],
                    "longitude": rectangle_low["longitude"]
                },
                "high": {
                    "latitude": rectangle_high["latitude"],
                    "longitude": rectangle_high["longitude"]
                }
            }
        },
        "pageSize": MAX_RESULTS_PER_PAGE,
        "languageCode": "de"
    }

    if page_token:
        body["pageToken"] = page_token

    time.sleep(SLEEP_BETWEEN_REQUESTS)
    api_call_count += 1

    try:
        response = requests.post(url, headers=headers, json=body, timeout=30)
        response.raise_for_status()
        data = response.json()

        places = data.get("places", [])
        next_token = data.get("nextPageToken", None)

        return places, next_token

    except requests.exceptions.HTTPError as e:
        log(f"  HTTP Error: {e}")
        log(f"  Response: {response.text[:500]}")
        return [], None
    except Exception as e:
        log(f"  Request Error: {e}")
        return [], None


def search_grid_cell(low, high, depth=0):
    """
    Durchsucht eine Grid-Zelle mit Pagination.
    Falls >= 60 Ergebnisse: rekursiv in 4 Unter-Zellen aufteilen.

    Args:
        low: dict mit latitude/longitude (SW-Ecke)
        high: dict mit latitude/longitude (NE-Ecke)
        depth: aktuelle Rekursionstiefe

    Returns:
        list: Alle gefundenen Places in dieser Zelle
    """
    indent = "  " * depth
    lat_range = f"{low['latitude']:.4f}-{high['latitude']:.4f}"
    lng_range = f"{low['longitude']:.4f}-{high['longitude']:.4f}"

    all_places = []
    page_token = None

    for page in range(MAX_PAGES):
        places, next_token = text_search_request(low, high, page_token)

        if not places and page == 0:
            log(f"{indent}Zelle [{lat_range}, {lng_range}]: 0 Ergebnisse")
            return []

        all_places.extend(places)

        if not next_token:
            break

        page_token = next_token

    count = len(all_places)
    log(f"{indent}Zelle [{lat_range}, {lng_range}]: {count} Ergebnisse (Tiefe {depth})")

    # Prüfe ob Verfeinerung nötig
    if count >= RESULTS_TRIGGER_REFINE and depth < MAX_RECURSION_DEPTH:
        log(f"{indent}  → {count} >= {RESULTS_TRIGGER_REFINE} Ergebnisse! Verfeinere in Unter-Zellen...")

        mid_lat = (low["latitude"] + high["latitude"]) / 2
        mid_lng = (low["longitude"] + high["longitude"]) / 2

        # 4 Quadranten
        sub_cells = [
            # SW
            ({"latitude": low["latitude"], "longitude": low["longitude"]},
             {"latitude": mid_lat, "longitude": mid_lng}),
            # SE
            ({"latitude": low["latitude"], "longitude": mid_lng},
             {"latitude": mid_lat, "longitude": high["longitude"]}),
            # NW
            ({"latitude": mid_lat, "longitude": low["longitude"]},
             {"latitude": high["latitude"], "longitude": mid_lng}),
            # NE
            ({"latitude": mid_lat, "longitude": mid_lng},
             {"latitude": high["latitude"], "longitude": high["longitude"]}),
        ]

        refined_places = []
        for sub_low, sub_high in sub_cells:
            refined_places.extend(search_grid_cell(sub_low, sub_high, depth + 1))

        return refined_places

    return all_places


def build_grid():
    """
    Erstellt das initiale Grid über Berlin.

    Returns:
        list of tuples: [(low_dict, high_dict), ...]
    """
    lat_step = (BERLIN_NORTH - BERLIN_SOUTH) / INITIAL_GRID_SIZE
    lng_step = (BERLIN_EAST - BERLIN_WEST) / INITIAL_GRID_SIZE

    # Overlap: 5% der Zellengröße an jeder Seite
    lat_overlap = lat_step * 0.05
    lng_overlap = lng_step * 0.05

    cells = []
    for row in range(INITIAL_GRID_SIZE):
        for col in range(INITIAL_GRID_SIZE):
            low = {
                "latitude": BERLIN_SOUTH + row * lat_step - lat_overlap,
                "longitude": BERLIN_WEST + col * lng_step - lng_overlap
            }
            high = {
                "latitude": BERLIN_SOUTH + (row + 1) * lat_step + lat_overlap,
                "longitude": BERLIN_WEST + (col + 1) * lng_step + lng_overlap
            }
            cells.append((low, high))

    return cells


def deduplicate_places(all_places):
    """
    Dedupliziert Places nach place_id (= places.id Feld).

    Args:
        all_places: Liste aller gesammelten Places

    Returns:
        list: Deduplizierte Places
    """
    seen = {}
    for place in all_places:
        place_id = place.get("id")
        if place_id and place_id not in seen:
            seen[place_id] = place

    return list(seen.values())


def sort_key_name(place):
    """Extrahiert den Namen für Sortierung."""
    return place.get("displayName", {}).get("text", "")


def main():
    global api_call_count

    log("=" * 60)
    log("Berlin Dönerläden Finder")
    log("=" * 60)

    if API_KEY == "DEIN_API_KEY_HIER":
        log("FEHLER: Bitte trage deinen Google API Key in der Variable API_KEY ein!")
        return

    # --------------------------------------------------------
    # Schritt 1: Grid erstellen
    # --------------------------------------------------------
    log(f"\nSchritt 1: Erstelle {INITIAL_GRID_SIZE}x{INITIAL_GRID_SIZE} Grid über Berlin")
    log(f"  Bounding Box: S={BERLIN_SOUTH}, N={BERLIN_NORTH}, W={BERLIN_WEST}, E={BERLIN_EAST}")

    grid_cells = build_grid()
    total_cells = len(grid_cells)
    log(f"  {total_cells} Grid-Zellen erstellt (mit 5% Overlap)")

    # --------------------------------------------------------
    # Schritt 2: Alle Grid-Zellen durchsuchen
    # --------------------------------------------------------
    log(f"\nSchritt 2: Durchsuche alle Grid-Zellen...")
    log(f"  Sleep zwischen Requests: {SLEEP_BETWEEN_REQUESTS}s")
    log(f"  Max Seiten pro Zelle: {MAX_PAGES}")
    log(f"  Verfeinerung ab: {RESULTS_TRIGGER_REFINE} Ergebnisse")
    log("")

    all_places_raw = []

    for i, (low, high) in enumerate(grid_cells):
        log(f"--- Zelle {i+1}/{total_cells} ---")
        places = search_grid_cell(low, high, depth=0)
        all_places_raw.extend(places)

        # Zwischenstand
        if (i + 1) % 10 == 0:
            unique_so_far = len(deduplicate_places(all_places_raw))
            log(f"  [Zwischenstand] {len(all_places_raw)} roh / {unique_so_far} unique nach {i+1}/{total_cells} Zellen")

    # --------------------------------------------------------
    # Schritt 3: Deduplizierung
    # --------------------------------------------------------
    log(f"\nSchritt 3: Deduplizierung")
    log(f"  Rohdaten: {len(all_places_raw)} Einträge")

    unique_places = deduplicate_places(all_places_raw)
    log(f"  Nach Deduplizierung: {len(unique_places)} einzigartige Dönerläden")

    # --------------------------------------------------------
    # Schritt 4: Ergebnisse speichern
    # --------------------------------------------------------
    log(f"\nSchritt 5: Ergebnisse speichern...")

    # Rohdaten komplett speichern (alle Attribute von der API)
    # Sortieren nach Name
    unique_places.sort(key=sort_key_name)

    output = {
        "metadata": {
            "timestamp": datetime.now().isoformat(),
            "area": "Berlin, Deutschland",
            "grid_size": f"{INITIAL_GRID_SIZE}x{INITIAL_GRID_SIZE}",
            "max_recursion_depth": MAX_RECURSION_DEPTH,
            "api_calls_total": api_call_count,
            "bounding_box": {
                "south": BERLIN_SOUTH,
                "north": BERLIN_NORTH,
                "west": BERLIN_WEST,
                "east": BERLIN_EAST
            }
        },
        "summary": {
            "grid_search_unique_count": len(unique_places),
            "grid_search_raw_count": len(all_places_raw)
        },
        "places": unique_places
    }

    with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
        json.dump(output, f, ensure_ascii=False, indent=2)

    log(f"  Gespeichert: {OUTPUT_FILE}")

    # --------------------------------------------------------
    # Zusammenfassung
    # --------------------------------------------------------
    log("\n" + "=" * 60)
    log("ZUSAMMENFASSUNG")
    log("=" * 60)
    log(f"  Grid-Suche (dedupliziert): {len(unique_places)} Dönerläden")
    log(f"  Grid-Suche (roh):          {len(all_places_raw)} Einträge")
    log(f"  API-Calls gesamt:          {api_call_count}")
    log(f"  Geschätzte Kosten:         ~${api_call_count * 0.032:.2f} (Text Search)")
    log("=" * 60)


if __name__ == "__main__":
    main()

[11:44:00] ============================================================
[11:44:00] Berlin Dönerläden Finder
[11:44:00] ============================================================
[11:44:00] 
Schritt 1: Erstelle 10x10 Grid über Berlin
[11:44:00]   Bounding Box: S=52.3382, N=52.6755, W=13.0884, E=13.7612
[11:44:00]   100 Grid-Zellen erstellt (mit 5% Overlap)
[11:44:00] 
Schritt 2: Durchsuche alle Grid-Zellen...
[11:44:00]   Sleep zwischen Requests: 2.0s
[11:44:00]   Max Seiten pro Zelle: 3
[11:44:00]   Verfeinerung ab: 60 Ergebnisse
[11:44:00] 
[11:44:00] --- Zelle 1/100 ---
[11:44:02] Zelle [52.3365-52.3736, 13.0850-13.1590]: 5 Ergebnisse (Tiefe 0)
[11:44:02] --- Zelle 2/100 ---
[11:44:04] Zelle [52.3365-52.3736, 13.1523-13.2263]: 0 Ergebnisse
[11:44:04] --- Zelle 3/100 ---
[11:44:06] Zelle [52.3365-52.3736, 13.2196-13.2936]: 0 Ergebnisse
[11:44:06] --- Zelle 4/100 ---
[11:44:09] Zelle [52.3365-52.3736, 13.2869-13.3609]: 3 Ergebnisse (Tiefe 0)
[11:44:09] --- Zelle 5/100 ---
[11:44:11] 

## Schritt 2 — Datenbereinigung

Die Google Places API gibt für jeden Ort bis zu **79 Felder** zurück, darunter viele verschachtelte (nested) JSON-Objekte. Dieser Schritt:

1. **Flacht** die nested Strukturen ab (`flatten_place()`): z.B. `location.latitude` → `latitude`, `displayName.text` → `displayName_text`
2. **Entfernt** irrelevante Spalten gemäß `COLUMNS_TO_DROP` (Telefonnummern, Icons, Parkplatz-Details, etc.)
3. **Bereinigt** Datentypen (Boolean-Spalten als nullable `pd.BooleanDtype()`, Ratings als float, Preise als Int64)
4. **Exportiert** das Ergebnis als `dataset_berlin_doener_clean.csv` (26 Spalten, 1.346 Zeilen)

Die auskommentierten Einträge in `COLUMNS_TO_DROP` zeigen alle verfügbaren Spalten — auskommentieren = Spalte wird **behalten**.

In [ ]:
"""
Berlin Dönerläden — Datenbereinigung & BI-Pipeline
====================================================
Liest die Rohdaten-JSON ein, zeigt alle Spalten, bereinigt,
flacht nested Strukturen ab und exportiert BI-ready.

Nutzung:
  1. Skript starten → zeigt alle Spalten
  2. COLUMNS_TO_DROP Liste anpassen
  3. Nochmal starten → erzeugt bereinigte Outputs
"""

import json
import pandas as pd
from pathlib import Path

# ============================================================
# Konfiguration
# ============================================================
INPUT_FILE = "dataset_berlin_doener.json"
OUTPUT_CSV = "dataset_berlin_doener_clean.csv"
# OUTPUT_JSON = "dataset_berlin_doener_clean.json"
# OUTPUT_SQLITE = "berlin_doener.db"
# SQLITE_TABLE = "doener_laeden"

"""
# ================================================================================
# SPALTENÜBERSICHT — 79 Spalten, 1346 Zeilen (generiert in vorheriger Ausführung)
# ================================================================================
#
#   Nr  Spalte                                        Typ            Missing  Sample
# ────  ───────────────────────────────────────────── ──────────── ─────────  ────────────────────────────────────────
#    1  id                                            object          0 ( 0.0%)  ChIJgRSVuEVFqEcR_67VANf0rwk
#    2  formattedAddress                              object          0 ( 0.0%)  Mariendorfer Damm 103, 12109 Berlin, Deutschland
#    3  shortFormattedAddress                         object          0 ( 0.0%)  Mariendorfer Damm 103, Berlin-Bezirk Tempelhof-Schöneberg
#    4  primaryType                                   object          0 ( 0.0%)  turkish_restaurant
#    5  nationalPhoneNumber                           object        516 (38.3%)  06564 908594938
#    6  internationalPhoneNumber                      object        516 (38.3%)  +49 6564 908594938
#    7  googleMapsUri                                 object          0 ( 0.0%)  https://maps.google.com/?cid=698045671534866175&g_mp=Cidnb29...
#    8  businessStatus                                object          0 ( 0.0%)  OPERATIONAL
#    9  rating                                        float64        13 ( 1.0%)  3.9
#   10  userRatingCount                               Int64          13 ( 1.0%)  269
#   11  iconMaskBaseUri                               object          0 ( 0.0%)  https://maps.gstatic.com/mapfiles/place_api/icons/v2/restaur...
#   12  iconBackgroundColor                           object          0 ( 0.0%)  #FF9E67
#   13  adrFormatAddress                              object          0 ( 0.0%)  <span class="street-address">Mariendorfer Damm 103</span>, <...
#   14  goodForChildren                               boolean       336 (25.0%)  True
#   15  goodForGroups                                 boolean       578 (42.9%)  True
#   16  delivery                                      boolean       240 (17.8%)  True
#   17  dineIn                                        boolean        24 ( 1.8%)  True
#   18  takeout                                       boolean        55 ( 4.1%)  True
#   19  servesLunch                                   boolean       115 ( 8.5%)  True
#   20  servesDinner                                  boolean       145 (10.8%)  True
#   21  servesBeer                                    boolean       530 (39.4%)  True
#   22  servesBrunch                                  boolean       523 (38.9%)  False
#   23  servesVegetarianFood                          boolean       555 (41.2%)  False
#   24  outdoorSeating                                boolean       549 (40.8%)  True
#   25  liveMusic                                     boolean       126 ( 9.4%)  False
#   26  restroom                                      boolean       537 (39.9%)  True
#   27  utcOffsetMinutes                              int64           0 ( 0.0%)  60
#   28  goodForWatchingSports                         boolean       583 (43.3%)  True
#   29  displayName_text                              object          0 ( 0.0%)  21 Dönerhaus Kebaphaus Pizzahaus
#   30  displayName_languageCode                      object          0 ( 0.0%)  de
#   31  latitude                                      float64         0 ( 0.0%)  52.4434554
#   32  longitude                                     float64         0 ( 0.0%)  13.386580299999999
#   33  viewport_low_latitude                         float64         0 ( 0.0%)  52.4420319197085
#   34  viewport_low_longitude                        float64         0 ( 0.0%)  13.385182069708499
#   35  viewport_high_latitude                        float64         0 ( 0.0%)  52.4447298802915
#   36  viewport_high_longitude                       float64         0 ( 0.0%)  13.387880030291504
#   37  primaryTypeDisplayName_text                   object          0 ( 0.0%)  Türkisches Restaurant
#   38  primaryTypeDisplayName_languageCode           object          0 ( 0.0%)  de
#   39  types                                         object          0 ( 0.0%)  turkish_restaurant,fast_food_restaurant,meal_takeaway,restau...
#   40  addressComponents                             object          0 ( 0.0%)  [{"longText": "103", "shortText": "103", "types": ["street_n...
#   41  plusCode_globalCode                           object          0 ( 0.0%)  9F4MC9VP+9J
#   42  plusCode_compoundCode                         object          0 ( 0.0%)  C9VP+9J Berlin, Deutschland
#   43  regularOpeningHours_openNow                   boolean        45 ( 3.3%)  True
#   44  regularOpeningHours_periods                   object         45 ( 3.3%)  [{"open": {"day": 0, "hour": 10, "minute": 30}, "close": {"d...
#   45  regularOpeningHours_weekdayDescriptions       object         45 ( 3.3%)  ["Montag: 10:30–02:00 Uhr", "Dienstag: 10:30–02:00 Uhr", "Mi...
#   46  currentOpeningHours_openNow                   boolean        45 ( 3.3%)  True
#   47  currentOpeningHours_periods                   object         45 ( 3.3%)  [{"open": {"day": 0, "hour": 10, "minute": 30, "date": {"yea...
#   48  currentOpeningHours_weekdayDescriptions       object         45 ( 3.3%)  ["Montag: 10:30–02:00 Uhr", "Dienstag: 10:30–02:00 Uhr", "Mi...
#   49  priceRange_start_currency                     object          0 ( 0.0%)  EUR
#   50  priceRange_start_units                        Int64          68 ( 5.1%)  1
#   51  priceRange_end_currency                       object          0 ( 0.0%)  EUR
#   52  priceRange_end_units                          Int64          68 ( 5.1%)  10
#   53  accessibility_wheelchairParking               boolean       810 (60.2%)  False
#   54  accessibility_wheelchairEntrance              boolean       718 (53.3%)  True
#   55  accessibility_wheelchairRestroom              boolean      1179 (87.6%)  True
#   56  accessibility_wheelchairSeating               boolean       829 (61.6%)  True
#   57  parking_freeLot                               boolean       891 (66.2%)  True
#   58  parking_paidLot                               boolean      1206 (89.6%)  True
#   59  parking_freeStreet                            boolean       951 (70.7%)  True
#   60  parking_paidStreet                            boolean      1168 (86.8%)  True
#   61  parking_valet                                 boolean      1298 (96.4%)  False
#   62  parking_freeGarage                            boolean      1291 (95.9%)  True
#   63  parking_paidGarage                            boolean      1301 (96.7%)  False
#   64  payment_creditCards                           boolean       927 (68.9%)  True
#   65  payment_debitCards                            boolean       717 (53.3%)  True
#   66  payment_cashOnly                              boolean       413 (30.7%)  False
#   67  payment_nfc                                   boolean       815 (60.5%)  True
#   68  editorialSummary_text                         object          0 ( 0.0%)  Kleiner, preiswerter italienischer Imbiss mit Selbstbedienun...
#   69  editorialSummary_languageCode                 object          0 ( 0.0%)  de
#   70  photos                                        object         20 ( 1.5%)  [{"name": "places/ChIJgRSVuEVFqEcR_67VANf0rwk/photos/ATCDNfX...
#   71  reviews                                       object         13 ( 1.0%)  [{"name": "places/ChIJgRSVuEVFqEcR_67VANf0rwk/reviews/Ci9DQU...
#   72  reservable                                    boolean       589 (43.8%)  False
#   73  menuForChildren                               boolean       667 (49.6%)  False
#   74  curbsidePickup                                boolean       856 (63.6%)  True
#   75  priceLevel                                    object        761 (56.5%)  PRICE_LEVEL_INEXPENSIVE
#   76  servesDessert                                 boolean       768 (57.1%)  False
#   77  servesWine                                    boolean       771 (57.3%)  False
#   78  websiteUri                                    object        786 (58.4%)  http://7daysoriginal.de/
#   79  servesBreakfast                               boolean      1028 (76.4%)  True
"""

# ======================================================================
# BLOCK 2 — COLUMNS_TO_DROP — Kommentar entfernen = Spalte droppen (generiert in vorheriger Ausführung)
# ======================================================================

COLUMNS_TO_DROP = [
    # "id",
    # "formattedAddress",
    # "shortFormattedAddress",
    # "primaryType",
    "nationalPhoneNumber",
    "internationalPhoneNumber",
    # "googleMapsUri",
    # "businessStatus",
    # "rating",
    # "userRatingCount",
    "iconMaskBaseUri",
    "iconBackgroundColor",
    "adrFormatAddress",
    "goodForChildren",
    "goodForGroups",
    # "delivery",
    # "dineIn",
    # "takeout",
    "servesLunch",
    "servesDinner",
    "servesBeer",
    "servesBrunch",
    "servesVegetarianFood",
    "outdoorSeating",
    # "liveMusic",
    "restroom",
    "utcOffsetMinutes",
    "goodForWatchingSports",
    # "displayName_text",
    "displayName_languageCode",
    # "latitude",
    # "longitude",
    "viewport_low_latitude",
    "viewport_low_longitude",
    "viewport_high_latitude",
    "viewport_high_longitude",
    # "primaryTypeDisplayName_text",
    "primaryTypeDisplayName_languageCode",
    # "types",
    # "addressComponents",
    "plusCode_globalCode",
    "plusCode_compoundCode",
    "regularOpeningHours_openNow",
    # "regularOpeningHours_periods",
    "regularOpeningHours_weekdayDescriptions",
    "currentOpeningHours_openNow",
    "currentOpeningHours_periods",
    "currentOpeningHours_weekdayDescriptions",
    # "priceRange_start_currency",
    # "priceRange_start_units",
    # "priceRange_end_currency",
    # "priceRange_end_units",
    "accessibility_wheelchairParking",
    "accessibility_wheelchairEntrance",
    "accessibility_wheelchairRestroom",
    "accessibility_wheelchairSeating",
    "parking_freeLot",
    "parking_paidLot",
    "parking_freeStreet",
    "parking_paidStreet",
    "parking_valet",
    "parking_freeGarage",
    "parking_paidGarage",
    "payment_creditCards",
    "payment_debitCards",
    "payment_cashOnly",
    "payment_nfc",
    # "editorialSummary_text",
    "editorialSummary_languageCode",
    # "photos",
    # "reviews",
    "reservable",
    "menuForChildren",
    "curbsidePickup",
    "priceLevel",
    "servesDessert",
    "servesWine",
    "websiteUri",
    "servesBreakfast"
]

# ============================================================
# Flatten-Logik: Nested JSON → flache Spalten
# ============================================================
def flatten_place(place: dict) -> dict:
    flat = {}

    simple_fields = [
        "id", "formattedAddress", "shortFormattedAddress",
        "primaryType", "nationalPhoneNumber", "internationalPhoneNumber",
        "websiteUri", "googleMapsUri", "businessStatus", "priceLevel",
        "rating", "userRatingCount", "iconMaskBaseUri", "iconBackgroundColor",
        "adrFormatAddress", "goodForChildren", "goodForGroups",
        "delivery", "dineIn", "takeout", "reservable",
        "servesBreakfast", "servesLunch", "servesDinner",
        "servesBeer", "servesWine", "servesBrunch",
        "servesVegetarianFood", "outdoorSeating", "liveMusic",
        "menuForChildren", "restroom", "utcOffsetMinutes",
        "servesDessert", "goodForWatchingSports", "curbsidePickup",
    ]
    for key in simple_fields:
        if key in place:
            flat[key] = place[key]

    dn = place.get("displayName", {})
    flat["displayName_text"] = dn.get("text", "")
    flat["displayName_languageCode"] = dn.get("languageCode", "")

    loc = place.get("location", {})
    flat["latitude"] = loc.get("latitude")
    flat["longitude"] = loc.get("longitude")

    vp = place.get("viewport", {})
    vp_low = vp.get("low", {})
    vp_high = vp.get("high", {})
    flat["viewport_low_latitude"] = vp_low.get("latitude")
    flat["viewport_low_longitude"] = vp_low.get("longitude")
    flat["viewport_high_latitude"] = vp_high.get("latitude")
    flat["viewport_high_longitude"] = vp_high.get("longitude")

    ptdn = place.get("primaryTypeDisplayName", {})
    flat["primaryTypeDisplayName_text"] = ptdn.get("text", "")
    flat["primaryTypeDisplayName_languageCode"] = ptdn.get("languageCode", "")

    flat["types"] = ",".join(place.get("types", []))

    if "addressComponents" in place:
        flat["addressComponents"] = json.dumps(place["addressComponents"], ensure_ascii=False)

    pc = place.get("plusCode", {})
    flat["plusCode_globalCode"] = pc.get("globalCode", "")
    flat["plusCode_compoundCode"] = pc.get("compoundCode", "")

    roh = place.get("regularOpeningHours", {})
    flat["regularOpeningHours_openNow"] = roh.get("openNow")
    flat["regularOpeningHours_periods"] = json.dumps(roh.get("periods", []), ensure_ascii=False) if roh.get("periods") else None
    flat["regularOpeningHours_weekdayDescriptions"] = json.dumps(roh.get("weekdayDescriptions", []), ensure_ascii=False) if roh.get("weekdayDescriptions") else None

    coh = place.get("currentOpeningHours", {})
    flat["currentOpeningHours_openNow"] = coh.get("openNow")
    flat["currentOpeningHours_periods"] = json.dumps(coh.get("periods", []), ensure_ascii=False) if coh.get("periods") else None
    flat["currentOpeningHours_weekdayDescriptions"] = json.dumps(coh.get("weekdayDescriptions", []), ensure_ascii=False) if coh.get("weekdayDescriptions") else None

    pr = place.get("priceRange", {})
    sp = pr.get("startPrice", {})
    ep = pr.get("endPrice", {})
    flat["priceRange_start_currency"] = sp.get("currencyCode", "")
    flat["priceRange_start_units"] = sp.get("units", "")
    flat["priceRange_end_currency"] = ep.get("currencyCode", "")
    flat["priceRange_end_units"] = ep.get("units", "")

    ao = place.get("accessibilityOptions", {})
    flat["accessibility_wheelchairParking"] = ao.get("wheelchairAccessibleParking")
    flat["accessibility_wheelchairEntrance"] = ao.get("wheelchairAccessibleEntrance")
    flat["accessibility_wheelchairRestroom"] = ao.get("wheelchairAccessibleRestroom")
    flat["accessibility_wheelchairSeating"] = ao.get("wheelchairAccessibleSeating")

    po = place.get("parkingOptions", {})
    flat["parking_freeLot"] = po.get("freeParkingLot")
    flat["parking_paidLot"] = po.get("paidParkingLot")
    flat["parking_freeStreet"] = po.get("freeStreetParking")
    flat["parking_paidStreet"] = po.get("paidStreetParking")
    flat["parking_valet"] = po.get("valetParking")
    flat["parking_freeGarage"] = po.get("freeGarageParking")
    flat["parking_paidGarage"] = po.get("paidGarageParking")

    pay = place.get("paymentOptions", {})
    flat["payment_creditCards"] = pay.get("acceptsCreditCards")
    flat["payment_debitCards"] = pay.get("acceptsDebitCards")
    flat["payment_cashOnly"] = pay.get("acceptsCashOnly")
    flat["payment_nfc"] = pay.get("acceptsNfc")

    es = place.get("editorialSummary", {})
    flat["editorialSummary_text"] = es.get("text", "")
    flat["editorialSummary_languageCode"] = es.get("languageCode", "")

    if "photos" in place:
        flat["photos"] = json.dumps(place["photos"], ensure_ascii=False)

    if "reviews" in place:
        flat["reviews"] = json.dumps(place["reviews"], ensure_ascii=False)

    return flat


# ============================================================
# Sample-Wert-Suche: bester nicht-leerer Wert über alle Zeilen
# ============================================================
def find_best_sample(series: pd.Series, max_len: int = 60) -> str:
    """
    Sucht den besten Sample-Wert für eine Spalte:
    - Überspringt None/NaN und leere Strings
    - Bevorzugt kurze, lesbare Werte
    - Kürzt lange Strings ab
    """
    for val in series.dropna():
        s = str(val).strip()
        if not s or s.lower() in ("nan", "none", "[]", "{}", '""', ""):
            continue
        if len(s) > max_len:
            s = s[:max_len] + "..."
        return s
    return "—"


# ============================================================
# Großer Spalten-Block
# ============================================================
def print_column_block(df: pd.DataFrame):
    total = len(df)
    lines = []
    lines.append('"""')
    lines.append("# ================================================================================")
    lines.append(f"# SPALTENÜBERSICHT — {df.shape[1]} Spalten, {total} Zeilen")
    lines.append("# ================================================================================")
    lines.append("#")
    lines.append(f"# {'Nr':>4}  {'Spalte':<45} {'Typ':<12} {'Missing':>9}  {'Sample'}")
    lines.append(f"# {'────':>4}  {'─'*45} {'─'*12} {'─'*9}  {'─'*40}")

    for i, col in enumerate(df.columns, 1):
        dtype = str(df[col].dtype)
        non_null = df[col].notna().sum()
        null_count = total - non_null
        null_pct = null_count / total * 100

        sample = find_best_sample(df[col])

        # Missing-Anzeige: Zahl + Prozent
        missing_str = f"{null_count:>4} ({null_pct:4.1f}%)"

        lines.append(f"# {i:>4}  {col:<45} {dtype:<12} {missing_str}  {sample}")

    lines.append('"""')

    block = "\n".join(lines)
    print(block)
    return block


# ============================================================
# COLUMNS_TO_DROP Template generieren
# ============================================================
def print_drop_block(df: pd.DataFrame):
    """
    Gibt einen fertigen COLUMNS_TO_DROP-Block aus.
    Alle Spalten sind auskommentiert (= werden NICHT gedroppt).
    Kommentar entfernen = Spalte wird gedroppt.
    Reihenfolge identisch zur Spaltenübersicht.
    """
    lines = []
    lines.append("COLUMNS_TO_DROP = [")
    for col in df.columns:
        lines.append(f"    # \"{col}\",")
    lines.append("]")

    block = "\n".join(lines)
    print(block)
    return block


# ============================================================
# Hauptprogramm
# ============================================================
def main():
    print("=" * 70)
    print("Berlin Dönerläden — Datenbereinigung & BI-Pipeline")
    print("=" * 70)

    input_path = Path(INPUT_FILE)
    if not input_path.exists():
        print(f"\nFEHLER: {INPUT_FILE} nicht gefunden!")
        return

    with open(INPUT_FILE, "r", encoding="utf-8") as f:
        data = json.load(f)

    metadata = data.get("metadata", {})
    summary  = data.get("summary", {})
    places_raw = data.get("places", data) if isinstance(data, dict) else data
    if isinstance(places_raw, dict):
        places_raw = list(places_raw.values())

    print(f"\nMetadaten:")
    print(f"  Zeitstempel:    {metadata.get('timestamp')}")
    print(f"  Grid:           {metadata.get('grid_size')}")
    print(f"  API-Calls:      {metadata.get('api_calls_total')}")
    print(f"\nSummary:")
    print(f"  Unique Places:  {summary.get('grid_search_unique_count')}")
    print(f"  Roh-Einträge:   {summary.get('grid_search_raw_count')}")
    print(f"\nAnzahl Places in JSON: {len(places_raw)}")

    # ── Schritt 2: Flatten ──
    print("\n" + "-" * 70)
    print("Schritt 2: Flatten der nested JSON-Strukturen")
    print("-" * 70)

    flattened = [flatten_place(p) for p in places_raw]
    df = pd.DataFrame(flattened)
    print(f"\nDataFrame Shape: {df.shape[0]} Zeilen × {df.shape[1]} Spalten")

    # ── Schritt 3: Spalten droppen ──
    if COLUMNS_TO_DROP:
        print("\n" + "-" * 70)
        print(f"Schritt 3: Lösche {len(COLUMNS_TO_DROP)} Spalten")
        print("-" * 70)
        existing_drops = [c for c in COLUMNS_TO_DROP if c in df.columns]
        missing_drops  = [c for c in COLUMNS_TO_DROP if c not in df.columns]
        if missing_drops:
            print(f"  WARNUNG: Spalten nicht gefunden: {missing_drops}")
        df = df.drop(columns=existing_drops)
        print(f"  Gelöscht: {existing_drops}")
        print(f"  Verbleibende Spalten: {df.shape[1]}")
    else:
        print("\n" + "-" * 70)
        print("Schritt 3: COLUMNS_TO_DROP leer — keine Spalten gelöscht")
        print("-" * 70)

    # ── Schritt 4: Datentypen bereinigen ──
    print("\n" + "-" * 70)
    print("Schritt 4: Datentypen bereinigen")
    print("-" * 70)

    for col in df.columns:
        if df[col].dropna().apply(lambda x: isinstance(x, bool)).all() and df[col].notna().any():
            df[col] = df[col].astype(pd.BooleanDtype())
            print(f"  {col}: → nullable bool (NA bleibt NA)")

    if "rating" in df.columns:
        df["rating"] = pd.to_numeric(df["rating"], errors="coerce")
    if "userRatingCount" in df.columns:
        df["userRatingCount"] = pd.to_numeric(df["userRatingCount"], errors="coerce").astype("Int64")
    if "priceRange_start_units" in df.columns:
        df["priceRange_start_units"] = pd.to_numeric(df["priceRange_start_units"], errors="coerce").astype("Int64")
    if "priceRange_end_units" in df.columns:
        df["priceRange_end_units"] = pd.to_numeric(df["priceRange_end_units"], errors="coerce").astype("Int64")

    print(f"\n  Finale Shape: {df.shape[0]} Zeilen × {df.shape[1]} Spalten")

    # ── Schritt 5: Export ──
    print("\n" + "-" * 70)
    print("Schritt 5: Export")
    print("-" * 70)

    df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
    print(f"  ✓ CSV:    {OUTPUT_CSV}")

    """
    df.to_json(OUTPUT_JSON, orient="records", force_ascii=False, indent=2)
    print(f"  ✓ JSON:   {OUTPUT_JSON}")

    import sqlite3
    conn = sqlite3.connect(OUTPUT_SQLITE)
    df.to_sql(SQLITE_TABLE, conn, if_exists="replace", index=False)
    cursor = conn.execute(f"SELECT COUNT(*) FROM {SQLITE_TABLE}")
    count = cursor.fetchone()[0]
    cursor = conn.execute(f"PRAGMA table_info({SQLITE_TABLE})")
    columns_info = cursor.fetchall()
    conn.close()
    print(f"  ✓ SQLite: {OUTPUT_SQLITE} (Tabelle: {SQLITE_TABLE}, {count} Zeilen, {len(columns_info)} Spalten)")
    """

    # ── Schritt 6: Großer Spalten-Block (zum Kopieren nach oben) ──
    print("\n\n")
    print("=" * 70)
    print("BLOCK 1 — SPALTENÜBERSICHT — direkt nach oben kopieren")
    print("=" * 70)
    print()
    print_column_block(df)

    # ── Schritt 7: COLUMNS_TO_DROP Template ──
    print("\n\n")
    print("=" * 70)
    print("BLOCK 2 — COLUMNS_TO_DROP — Kommentar entfernen = Spalte droppen")
    print("=" * 70)
    print()
    print_drop_block(df)


if __name__ == "__main__":
    main()

Berlin Dönerläden — Datenbereinigung & BI-Pipeline

Metadaten:
  Zeitstempel:    2026-03-18T13:04:36.204001
  Grid:           10x10
  API-Calls:      184

Summary:
  Unique Places:  1346
  Roh-Einträge:   1581

Anzahl Places in JSON: 1346

----------------------------------------------------------------------
Schritt 2: Flatten der nested JSON-Strukturen
----------------------------------------------------------------------

DataFrame Shape: 1346 Zeilen × 79 Spalten

----------------------------------------------------------------------
Schritt 3: Lösche 53 Spalten
----------------------------------------------------------------------
  Gelöscht: ['nationalPhoneNumber', 'internationalPhoneNumber', 'iconMaskBaseUri', 'iconBackgroundColor', 'adrFormatAddress', 'goodForChildren', 'goodForGroups', 'servesLunch', 'servesDinner', 'servesBeer', 'servesBrunch', 'servesVegetarianFood', 'outdoorSeating', 'restroom', 'utcOffsetMinutes', 'goodForWatchingSports', 'displayName_languageCode', 'viewpo